# Phishing Detection: A Classical Baseline
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/01_baseline.ipynb)

In this series we build a phishing filter from a language model, attack it, defend it, and deploy it. A new model needs something to beat, so we start with a classical one.

Run the cells in order and answer the questions in the **Your answer** cells.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

from seclab import config, data, metrics, notebook
notebook.banner()

## The data

The dataset has about 18,000 labeled emails. Many of the safe emails come from Enron, an energy company whose internal email became public after it collapsed in 2001. Keep that in mind.

In [ ]:
df = data.load_emails()
print(df.shape)
df["label"].value_counts()

**Python note.** `df` is a pandas DataFrame, a table with named columns. `df["label"]` is one column. Label 1 means phishing and 0 means safe. A cell's last line is displayed automatically, so there's no need to call `print` on it.

In [ ]:
print(df[df["label"] == 1]["text"].iloc[0])

`load_emails` drops empty rows and exact duplicates and keeps the first 1,000 characters of each email.

**Question 1.** Why do duplicates have to be removed *before* we split the data into training and test sets?

**Your answer:**

## Train, validation and test

We train on the training set, make choices (like the threshold) on the validation set, and touch the test set only to report results. Every lab in this series uses the same split and the same test emails.

In [ ]:
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
print(len(train), len(val), len(test), len(test_small))

## TF-IDF features and logistic regression

Each email becomes a vector with one entry per word or word pair. The entry for term $t$ in email $d$ is

$$\text{tfidf}(t,d) = \big(1 + \ln \text{count}(t,d)\big)\cdot\left(\ln\frac{1+N}{1+\text{df}(t)} + 1\right)$$

where $N$ is the number of emails and $\text{df}(t)$ is the number of emails that contain $t$. A word that appears in every email gets a small weight. A rare word gets a large one. Each vector is then scaled to length 1.

Logistic regression turns the vector $x$ into a probability:

$$p(\text{phishing}\mid x) = \sigma(w^\top x + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train["text"])
print(X_train.shape)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train["label"])
p_test = clf.predict_proba(vec.transform(test_small["text"]))[:, 1]

**Python note.** `fit_transform` learns the vocabulary from the training emails and converts them. For any other emails we call only `transform`, because learning from test data would be cheating. `[:, 1]` takes column 1 of a 2D array, which holds the probability of phishing.

## How good is it?

Recall is the share of phishing emails we catch. The false positive rate (FPR) is the share of safe emails we wrongly block.

In [ ]:
m = metrics.evaluate(test_small["label"], p_test)
m

**Question 2.** For a bank's email filter, which mistake costs more, a false negative or a false positive? Say who pays for each one.

**Your answer:**

## What did it learn?

In [ ]:
import numpy as np

words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print("Most safe terms:     ", ", ".join(words[order[:20]]))
print("Most phishing terms: ", ", ".join(words[order[-20:]]))

**Question 3.** Look at the most safe terms. Are they signs of a safe email, or signs of where the safe emails came from? What would happen if this filter ran on another company's email?

**Your answer:**

## Choosing a threshold

In [ ]:
import pandas as pd

p_val = clf.predict_proba(vec.transform(val["text"]))[:, 1]
rows = []
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    m_t = metrics.evaluate(val["label"], p_val, threshold=t)
    rows.append({"threshold": t, "recall": m_t["recall"], "fpr": m_t["fpr"], "precision": m_t["precision"]})
pd.DataFrame(rows).round(3)

**Python note.** `for t in [...]` loops over a list, like Java's enhanced for loop. `{"threshold": t, ...}` is a dictionary, Python's version of a `HashMap`.

**Question 4.** Using your answer to Question 2, pick a threshold for the bank. Why that one?

**Your answer:**

Save the result so later labs can compare against it.

In [ ]:
notebook.save_result("TF-IDF baseline", m)